In [2]:
import sys
import os

sys.path.append(
    os.path.abspath("..")
)

In [3]:
import torch
import pandas as pd
import networkx as nx

from torch_geometric.utils import from_networkx
from torch_geometric.data import Data

from src.preprocessing import DataPreprocessor
from src.feature_engineering import FeatureEngineer
from src.graph_builder import GraphBuilder
from src.gnn_models import GraphSAGE, GNNTrainer


In [4]:
DATA_PATH = "../data/raw/delivery_data.csv"

preprocessor = DataPreprocessor(DATA_PATH)

engineer = FeatureEngineer()

graph_builder = GraphBuilder()

trainer = GNNTrainer()

df = preprocessor.load_data()

df = preprocessor.clean_data(df)

df = engineer.create_temporal_features(df)

df = engineer.create_targets(df)

df = preprocessor.encode_features(df)


Loaded dataset shape: (144867, 24)


In [5]:
corridor_df = engineer.build_corridor_features(df)

print(corridor_df.head())


   source_center  destination_center  mean_delay_ratio  delay_variance  \
0              0                 562          2.678377        1.291174   
1              1                1190          6.285714        3.835948   
2              2                1341          1.924721        0.200800   
3              3                 652          4.531673        1.998161   
4              3                 653          3.347826        1.275992   

   avg_distance  avg_actual_time  sla_breach_rate  trip_count  
0     20.911341        63.702703              1.0          37  
1     12.518075        78.250000              1.0           4  
2     36.861711        50.555556              1.0          18  
3     52.518967       181.666667              1.0           3  
4     45.696067       122.333333              1.0           3  


In [6]:
G = graph_builder.build_graph(corridor_df)

print(G)

DiGraph with 1500 nodes and 2767 edges


In [7]:

metrics_df = graph_builder.compute_graph_metrics(G)

print(metrics_df.head())


     node  in_degree  out_degree  betweenness  pagerank
0     0.0          1           1     0.000007  0.000118
1   562.0         23           2     0.095830  0.006769
2     1.0          1           1     0.090439  0.001090
3  1190.0          2           1     0.091480  0.001042
4     2.0          2           1     0.018325  0.000756


In [8]:
node_mapping = {
    node: idx
    for idx, node in enumerate(G.nodes())
}

edges = []

for u, v in G.edges():

    edges.append([
        node_mapping[u],
        node_mapping[v]
    ])

edge_index = torch.tensor(edges).t().contiguous()

In [9]:
node_features = metrics_df[
    [
        "in_degree",
        "out_degree",
        "betweenness",
        "pagerank"
    ]
].values

x = torch.tensor(
    node_features,
    dtype=torch.float
)

In [10]:
y = torch.tensor(
    metrics_df["betweenness"].values,
    dtype=torch.float
)

graph_data = Data(
    x=x,
    edge_index=edge_index,
    y=y
)

print(graph_data)

Data(x=[1500, 4], edge_index=[2, 2767], y=[1500])


In [11]:
model = GraphSAGE(
    input_dim=x.shape[1],
    hidden_dim=16,
    output_dim=1
)

In [12]:
trainer.train_model(
    model,
    graph_data,
    epochs=100
)

print("GNN Training Completed")

Epoch 0 | Loss: 0.6163
Epoch 10 | Loss: 0.0736
Epoch 20 | Loss: 0.0297
Epoch 30 | Loss: 0.0123
Epoch 40 | Loss: 0.0066
Epoch 50 | Loss: 0.0049
Epoch 60 | Loss: 0.0029
Epoch 70 | Loss: 0.0022
Epoch 80 | Loss: 0.0017
Epoch 90 | Loss: 0.0014
GNN Training Completed
